# Engineering Orchestrator

> Notebook generated from [`examples/subgraphs/11_engineering_orchestrator.py`](../../examples/subgraphs/11_engineering_orchestrator.py).

| Item | Value |
|------|-------|
| **Dataset** | GitHub Issues (local CSV) |
| **API key** | ✅ **No API keys required** — uses stubs/mocks. |

**Expected result:** Simulated routing + real LangGraph with stubs. Supervisor → coder|codeact|planner|file_manager|skill_manager.

---

## Setup

```bash
uv pip install -e ".[dev,all]"
```

> To use `await main()` directly in cells, this notebook
> installs `nest_asyncio` in the first cell.

---

## Original docstring

```
Engineering Orchestrator — Hierarchical Engineering Domain Orchestrator
=============================================================================
Subgraph: prismal.agents.subgraphs.engineering_orchestrator

Dataset: GitHub Issues (LangChain) — real engineering requests
  • `Langgraph_tutorials/data/github-issues/github_issues.csv` contains
    open issues from the `langchain-ai/langchain` repo. We filter issues
    with non-empty `body` and reclassify them to the 5 domain leaves:
      - coder           : "fix the bug in", "implement feature"
      - codeact         : "this code throws", "run this script and"
      - planner         : "design a new", "what's the architecture for"
      - file_manager    : "rename / move / locate file"
      - skill_manager   : "install / activate / build skill"
  • Why: GitHub Issues are real engineering requests with
    natural phrasing — perfect for testing the routing of the
    `engineering_supervisor` (SPEC-042 / Phase 40).

Topology:
    engineering_supervisor (domain LLM router)
      ├──► coder           (ReAct + tools)
      ├──► codeact         (generates and runs Python in a sandbox)
      ├──► planner         (SDD decomposition / specs)
      ├──► file_manager    (filesystem operations)
      └──► skill_manager   (install/activate skills)
    each leaf → engineering_supervisor (loop-breaker → END)

Demo modes:
  1. demo_simulation()      — keyword-based routing (no LLM)
  2. demo_real_langgraph()  — real graph with stubs and MemorySaver

Usage:
    uv run python examples/subgraphs/11_engineering_orchestrator.py
```

In [ ]:
# Enable nested event loop to use `await` directly in Jupyter.
import sys
if 'nest_asyncio' not in sys.modules:
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        %pip install -q nest_asyncio
        import nest_asyncio
        nest_asyncio.apply()

## Locate the dataset

This notebook reads data from `data/` at the repo root. The cell below searches for that folder by walking up from the current cwd.

In [ ]:
# Resolve the repo's data/ folder by walking up until it is found.
# Replaces the DATA_ROOT
# that the original script had when it lived inside prismal/examples/.
import pathlib


def _find_data_root() -> pathlib.Path:
    cur = pathlib.Path.cwd().resolve()
    for _ in range(8):
        if (cur / "data").is_dir():
            return cur / "data"
        if cur.parent == cur:
            break
        cur = cur.parent
    raise FileNotFoundError(
        "data/ folder not found. Run the notebook from the "
        "prisma-notebooks/ repo where data/ is at the root."
    )


DATA_ROOT = _find_data_root()
print(f"DATA_ROOT = {DATA_ROOT}")

## Setup · imports and dataset

In [ ]:
from __future__ import annotations

import asyncio
import csv
import re
import textwrap
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

## Dataset: GitHub Issues reclassified into the 5 domain leaves

In [ ]:
GH_ISSUES = (
    DATA_ROOT
    / "github-issues"
    / "github_issues.csv"
)

ENGINEERING_LEAVES = ("coder", "codeact", "planner", "file_manager", "skill_manager")


@dataclass
class EngTask:
    """Engineering request derived from a GitHub Issue."""

    id: str
    title: str
    request: str
    expected_leaf: str
    complexity: str = "MEDIUM"


def _classify(title: str, body: str) -> str:
    """Demo heuristic to assign the expected leaf based on content."""
    text = f"{title} {body}".lower()
    if re.search(r"\b(spec|design|architecture|prd|plan)\b", text):
        return "planner"
    if re.search(r"\b(file|path|rename|move|directory|workspace)\b", text):
        return "file_manager"
    if re.search(r"\b(skill|install|activate|register|plugin)\b", text):
        return "skill_manager"
    if re.search(r"\b(traceback|run this|execute|script|notebook|stdout)\b", text):
        return "codeact"
    return "coder"


def _embedded_tasks() -> list[EngTask]:
    """Fallback plan if the CSV is not available."""
    return [
        EngTask("ENG-01", "Fix TypeError in chat_models.py",
                "There's a TypeError when calling .invoke() with None input. "
                "Need to add a guard and a regression test.",
                "coder", "MEDIUM"),
        EngTask("ENG-02", "Implement new YAML parser",
                "Run this script and capture stdout — the parser should accept "
                "multi-document streams.",
                "codeact", "HIGH"),
        EngTask("ENG-03", "Design new retrieval architecture",
                "We need a spec for the new hybrid retrieval architecture with "
                "BM25 + dense + reranking. Write a PRD.",
                "planner", "HIGH"),
        EngTask("ENG-04", "Rename `utils.py` to `helpers.py`",
                "Move every reference and update imports across the package.",
                "file_manager", "LOW"),
        EngTask("ENG-05", "Install pdf-export skill",
                "We need to register and activate the pdf-export skill for "
                "the docs team.",
                "skill_manager", "LOW"),
        EngTask("ENG-06", "Bug: race condition in cache layer",
                "The cache returns stale entries under concurrent writes. "
                "Reproduce, isolate, and patch.",
                "coder", "HIGH"),
    ]


def _load_tasks(limit: int = 8) -> list[EngTask]:
    """Load issues from CSV and map them to EngTask with expected leaf."""
    if not GH_ISSUES.exists():
        return _embedded_tasks()

    out: list[EngTask] = []
    try:
        with GH_ISSUES.open(encoding="utf-8") as fh:
            for i, row in enumerate(csv.DictReader(fh)):
                body = (row.get("body") or "").strip()
                title = (row.get("title") or "").strip()
                if not body or len(body) < 30:
                    continue
                out.append(
                    EngTask(
                        id=f"GH-{i:04d}",
                        title=title[:80],
                        request=textwrap.shorten(body, 220),
                        expected_leaf=_classify(title, body),
                    )
                )
                if len(out) >= limit:
                    break
    except Exception:  # noqa: BLE001 - corrupt dataset → fallback
        return _embedded_tasks()
    return out or _embedded_tasks()

## engineering_supervisor simulation (no LLM)

In [ ]:
def simulate_supervisor(request: str) -> str:
    """Domain-supervisor mock: same heuristic used when building the dataset."""
    return _classify(request, request)

## Leaf stubs

In [ ]:
LEAF_ICONS = {
    "coder": "🧑‍💻",
    "codeact": "🐍",
    "planner": "📐",
    "file_manager": "📁",
    "skill_manager": "🛠 ",
}


@dataclass
class LeafResult:
    leaf: str
    summary: str
    artifacts: list[str] = field(default_factory=list)


def stub_coder(t: EngTask) -> LeafResult:
    return LeafResult("coder",
        f"Patched issue {t.id}. Added regression test and updated CHANGELOG.",
        ["patch.diff", "test_regression.py"])


def stub_codeact(t: EngTask) -> LeafResult:
    return LeafResult("codeact",
        f"Executed Python plan for {t.id}; produced expected stdout and validated outputs.",
        ["plan.py", "stdout.log"])


def stub_planner(t: EngTask) -> LeafResult:
    return LeafResult("planner",
        f"Drafted PRD + SDD spec for {t.id}; identified 4 milestones and 6 risks.",
        ["spec.md", "milestones.png"])


def stub_file_manager(t: EngTask) -> LeafResult:
    return LeafResult("file_manager",
        f"Renamed and re-wired imports for {t.id}; 17 files changed under workspace.",
        ["rename.log"])


def stub_skill_manager(t: EngTask) -> LeafResult:
    return LeafResult("skill_manager",
        f"Installed + activated skill referenced by {t.id}; smoke test passed.",
        ["skills/active/*"])


LEAVES = {
    "coder": stub_coder,
    "codeact": stub_codeact,
    "planner": stub_planner,
    "file_manager": stub_file_manager,
    "skill_manager": stub_skill_manager,
}

## Demo 1: pure simulation

In [ ]:
def demo_simulation(tasks: list[EngTask]) -> None:
    print("\n" + "=" * 72)
    print(" Demo 1 · Routing simulation (no LLM, no LangGraph)")
    print("=" * 72)

    hits = 0
    for t in tasks:
        routed = simulate_supervisor(t.request)
        ok = routed == t.expected_leaf
        hits += int(ok)
        result = LEAVES[routed](t)
        print(f"\n  {t.id} · {t.title[:60]}")
        print(f"    supervisor → {LEAF_ICONS[routed]}{routed}    "
              f"(expected={t.expected_leaf})  {'✓' if ok else '✗'}")
        print(f"    {result.summary[:90]}")
    pct = 100.0 * hits / len(tasks)
    print(f"\n  Routing accuracy: {hits}/{len(tasks)} ({pct:.1f}%)")

## Demo 2: real LangGraph with stubs

In [ ]:
async def demo_real_langgraph(tasks: list[EngTask]) -> None:
    print("\n" + "=" * 72)
    print(" Demo 2 · Real LangGraph with SubgraphFactory + MemorySaver + stubs")
    print("=" * 72)
    try:
        from typing import Annotated, TypedDict

        from langchain_core.messages import AIMessage, HumanMessage
        from langgraph.checkpoint.memory import MemorySaver

        from prismal.langgraph import StateGraph, add_messages

        from prismal.agents.subgraphs.engineering_orchestrator.builder import (
            ENGINEERING_AGENTS,
            _engineering_router,
        )
    except ImportError as exc:
        print(f"  ⚠  missing dependency: {exc}")
        return

    class DemoState(TypedDict, total=False):
        messages: Annotated[list, add_messages]
        current_agent: str
        next_agent: str | None
        metadata: dict
        session_id: str

    def _make_leaf_node(name: str):
        async def _node(state: DemoState) -> dict[str, Any]:
            task_id = state.get("metadata", {}).get("task_id", "?")
            return {
                "messages": [AIMessage(content=f"[{name}] handled {task_id}", name=name)],
                "current_agent": name,
            }
        _node.__name__ = name
        return _node

    async def demo_supervisor(state: DemoState) -> dict[str, Any]:
        msgs = list(state.get("messages") or [])
        if msgs and getattr(msgs[-1], "type", "") == "ai" \
                and state.get("current_agent") in ENGINEERING_AGENTS:
            return {"current_agent": "engineering_supervisor", "next_agent": None}
        human = next((m for m in reversed(msgs) if getattr(m, "type", "") == "human"), None)
        request = human.content if human else ""
        return {
            "current_agent": "engineering_supervisor",
            "next_agent": simulate_supervisor(request),
        }

    sg = StateGraph(DemoState)
    sg.add_node("engineering_supervisor", demo_supervisor)
    for leaf in ENGINEERING_AGENTS:
        sg.add_node(leaf, _make_leaf_node(leaf))
        sg.add_edge(leaf, "engineering_supervisor")
    sg.set_entry_point("engineering_supervisor")
    sg.add_conditional_edges("engineering_supervisor", _engineering_router)
    compiled = sg.compile(checkpointer=MemorySaver())

    for t in tasks[:4]:
        config = {"configurable": {"thread_id": t.id}}
        state = {
            "messages": [HumanMessage(content=t.request)],
            "metadata": {"task_id": t.id},
        }
        final = await compiled.ainvoke(state, config=config)
        last = final["messages"][-1]
        print(f"\n  {t.id} → {last.name}: {last.content}")


async def main() -> None:
    tasks = _load_tasks(limit=10)
    print(f"Loaded {len(tasks)} engineering tasks ({'dataset' if GH_ISSUES.exists() else 'fallback'})")
    demo_simulation(tasks)
    await demo_real_langgraph(tasks)


if __name__ == "__main__":
    asyncio.run(main())

---

## Run the demo

The original file ends with `asyncio.run(main())`. In Jupyter use:

```python
await main()
```

Thanks to the `nest_asyncio.apply()` in the first cell, the
`async` calls run without needing a separate event loop.

In [ ]:
# Uncomment to run the end-to-end demo (requires API key if applicable).
# await main()